# V15 · 原生15分钟与5分钟退出的保存账本复核

## tl;dr
三格普通Python完成保存账本核对：251病例、462控制、原154组三对照。
原5m完成交易251笔、净均值-14.306631864989603bp；
原生15m完成交易251笔、净均值-16.662360573261154bp。
D已知251/251，均值-2.3557287082715495bp；
I已知154/251，均值0.5946002624257842bp。
未知没有补零；D在相同已知配对上核对，不用不同完成集合的均值直接相减。
均值比较不等于盈利确认，原生规格也不是纯时钟差异。
复用同一验证器，未重做原生上下文、原始价格或p值；Jupyter与完整schema仍未验证。

## Context & Methods
原1h大实体/吞没穿SMA40的直接入场保持不变，不加4h或前20小时入场门。
比较原生5m SMA40(HL2)真实同向→反向退出与原生15m同规则；K1极值硬止损、
原72小时、已完成交易20bp往返假设不变。原生规格同时改变聚合、初始可用颜色
和均线记忆（3小时20分钟→10小时），不能解释为只改变检查频率。

### Key Assumptions
本notebook只复核保存的成交/母/匹配/单仓/差值账本及其相互一致性。
不重复native_entry_context与原V5上下文联结，不重算SMA、首次翻色或原始5m路径；
原生上下文是否通过正式验证应查对应验证收据，不由这份notebook替代。
不重算推断p；反复研究的2023–2024及154/251固定匹配支持不能提供独立盈利确认。

## Data
固定summary SHA，读取两臂各六表和三个delta，共十五份保存CSV；逐一校验output_hashes。
V15验证器及V12/V11公共stdlib辅助脚本同时固定SHA，仅调用verify_tables纯表函数，
不调用整仓verify/main，不加载历史原始价格或其他实验结果。
从仓库运行或设置NOTEBOOK_REPOSITORY_ROOT；执行代码格只需Python标准库。

In [1]:
import csv, gzip, hashlib, importlib.util, io, json
from pathlib import Path
RESULTS_RELATIVE='experiments/active/exp-btcusdtp-1h-native15-exit-preholdout-20260906-v15/results'
EVIDENCE_FILES=('baseline/case_trades.csv.gz', 'baseline/control_trades.csv.gz', 'baseline/case_episodes.csv.gz', 'baseline/control_episodes.csv.gz', 'baseline/matched.csv', 'baseline/single_pending.csv.gz', 'candidate/case_trades.csv.gz', 'candidate/control_trades.csv.gz', 'candidate/case_episodes.csv.gz', 'candidate/control_episodes.csv.gz', 'candidate/matched.csv', 'candidate/single_pending.csv.gz', 'case_delta.csv', 'excess_delta.csv', 'serial_delta.csv')
TABLE_FILES={'case_trades': 'case_trades.csv.gz', 'control_trades': 'control_trades.csv.gz', 'case_episodes': 'case_episodes.csv.gz', 'control_episodes': 'control_episodes.csv.gz', 'matched': 'matched.csv', 'single_pending': 'single_pending.csv.gz'}
DELTA_NAMES=('case_delta', 'excess_delta', 'serial_delta')
VERIFIER_FILES=('scripts/verify_hourly_impulse_native_exit_v15.py', 'scripts/verify_hourly_impulse_frozen_ma_v12.py', 'scripts/verify_hourly_impulse_launch_v11.py')
SUMMARY_SHA256='d27f820b26b6bb2dd56bd7b1eba5017907656ea70dc7bd08cbe5c1c1c14f80d4'
VERIFIER_HASHES={'scripts/verify_hourly_impulse_native_exit_v15.py': '2c0b6bc4c10b8fbd8474052e942bfccdc8590b6bef796248c6b64d2885b4bc69', 'scripts/verify_hourly_impulse_frozen_ma_v12.py': 'b71d38418cb3d166a32c9f974c6c60f6a3121eb90583f88da072ea6717e21939', 'scripts/verify_hourly_impulse_launch_v11.py': '2b12c8309301bf2c7960679838ee048c30960a353317a2454842f2ad6c892362'}
def require(ok,message):
    if not ok:raise ValueError(message)
def digest(data):return hashlib.sha256(data).hexdigest()
hint=globals().get("NOTEBOOK_REPOSITORY_ROOT")
roots=[Path(hint)] if hint is not None else [Path.cwd(),*Path.cwd().parents]
root=next((p.resolve() for p in roots if (p/RESULTS_RELATIVE/"summary.json").is_file()),None)
require(root is not None,"Run from repository or set NOTEBOOK_REPOSITORY_ROOT")
directory=(root/RESULTS_RELATIVE).resolve()
require(directory.is_relative_to(root),"Evidence escaped repository")
def evidence_path(name):
    require(name in ("summary.json",*EVIDENCE_FILES),"Evidence not allowlisted")
    path=(directory/name).resolve()
    require(path==directory/name,"Evidence symlink changed fixed identity")
    return path
def verifier_path(name):
    require(name in VERIFIER_FILES,"Verifier not allowlisted")
    path=(root/name).resolve()
    require(path==root/name,"Verifier symlink changed identity")
    return path
print("Saved-ledger evidence only:",RESULTS_RELATIVE)

Saved-ledger evidence only: experiments/active/exp-btcusdtp-1h-native15-exit-preholdout-20260906-v15/results


### 1. 固定来源和两臂唯一管理差异

In [2]:
payload=evidence_path("summary.json").read_bytes()
require(digest(payload)==SUMMARY_SHA256,"Pinned summary hash mismatch")
def reject_constant(value):raise ValueError("Nonfinite JSON: "+value)
summary=json.loads(payload,parse_constant=reject_constant)
require(summary["experiment_id"]=='exp-btcusdtp-1h-native15-exit-preholdout-20260906-v15',"Wrong V15 experiment")
require(summary["status"]=="diagnostic_only_no_candidate_acceptance","Unexpected acceptance claim")
for flag in ("holdout_consumed","audit_prices_loaded","training_eligible","production_eligible","all_financial_gates_pass"):
    require(summary[flag] is False,"Unexpected safety/eligibility flag: "+flag)
require(abs(summary["known_coverage_ceiling"]-154/251)<1e-12,"Original matching support changed")
expected_policies={"baseline":{'id': '5m_native40', 'management_minutes': 5, 'ma_kind': 'SMA', 'ma_length': 40, 'exit_mode': 'transition_colour', 'confirmations': 1},"candidate":{'id': '15m_native40', 'management_minutes': 15, 'ma_kind': 'SMA', 'ma_length': 40, 'exit_mode': 'transition_colour', 'confirmations': 1}}
require(set(summary["arms"])==set(expected_policies),"Wrong arms")
for arm,policy in expected_policies.items():
    require(json.dumps(summary["arms"][arm]["policy"],sort_keys=True)==json.dumps(policy,sort_keys=True),"Native management policies changed")
loaded={}
for name in EVIDENCE_FILES:
    data=evidence_path(name).read_bytes()
    require(digest(data)==summary["output_hashes"][name],"CSV hash mismatch: "+name)
    text=gzip.decompress(data).decode() if name.endswith(".gz") else data.decode()
    reader=csv.DictReader(io.StringIO(text))
    require(reader.fieldnames and len(reader.fieldnames)==len(set(reader.fieldnames)),"Invalid CSV headers")
    rows=list(reader)
    require(all(None not in r and all(v is not None for v in r.values()) for r in rows),"Malformed CSV")
    loaded[name]=rows
for name in VERIFIER_FILES:
    require(digest(verifier_path(name).read_bytes())==VERIFIER_HASHES[name],"Verifier dependency hash mismatch: "+name)
tables={arm:{key:loaded[arm+"/"+file] for key,file in TABLE_FILES.items()} for arm in ("baseline","candidate")}
tables.update({key:loaded[key+".csv"] for key in DELTA_NAMES})
print("Pinned summary,",len(loaded),"saved CSVs and all three verifier modules verified")

Pinned summary, 15 saved CSVs and all three verifier modules verified


## Results

### 2. 复用固定纯表验证器，保留完整配对分母

In [3]:
spec=importlib.util.spec_from_file_location("_v15_notebook_saved_verifier",verifier_path(VERIFIER_FILES[0]))
verifier=importlib.util.module_from_spec(spec)
spec.loader.exec_module(verifier)
validation=verifier.verify_tables(tables,summary["arms"],summary["effects"])
require(isinstance(validation,dict) and validation.get("status","passed")=="passed","Failed validation receipt")
require(validation.get("counts")=={"cases":251,"controls":462,"matched":154,"unmatched":97},"Verifier did not retain full population")
require(set(validation.get("effects",{}))==set(DELTA_NAMES),"Verifier omitted a paired effect")
verified={"counts":validation["counts"],"effects":validation["effects"],
    "baseline_mean_net_bp":summary["arms"]["baseline"]["metrics"]["mean_net_bp"],
    "candidate_mean_net_bp":summary["arms"]["candidate"]["metrics"]["mean_net_bp"],
    "baseline_events":summary["arms"]["baseline"]["metrics"]["events"],
    "candidate_events":summary["arms"]["candidate"]["metrics"]["events"],
    "raw_price_replay":False,"native_context_reverified":False,"inferential_p_recomputed":False,
    "verifier_reused_not_independent":True}
print("Verified saved financial ledgers:",json.dumps(verified,ensure_ascii=False,allow_nan=False))
print("Same pinned verifier reused. No native context, raw price or inferential-p recomputation here.")

Verified saved financial ledgers: {"counts": {"cases": 251, "controls": 462, "matched": 154, "unmatched": 97}, "effects": {"case_delta": {"total_pairs": 251, "n": 251, "unknown_pairs": 0, "improved": 79, "worsened": 157, "unchanged": 15, "mean_bp": -2.3557287082715495, "sum_event_bp": -591.2879057761588}, "excess_delta": {"total_pairs": 251, "n": 154, "unknown_pairs": 97, "improved": 79, "worsened": 75, "unchanged": 0, "mean_bp": 0.5946002624257842, "sum_event_bp": 91.56844041357077}, "serial_delta": {"total_pairs": 251, "n": 251, "unknown_pairs": 0, "improved": 78, "worsened": 158, "unchanged": 15, "mean_bp": -2.6191044538915693, "sum_event_bp": -657.3952179267839}}, "baseline_mean_net_bp": -14.306631864989603, "candidate_mean_net_bp": -16.662360573261154, "baseline_events": 251, "candidate_events": 251, "raw_price_replay": false, "native_context_reverified": false, "inferential_p_recomputed": false, "verifier_reused_not_independent": true}
Same pinned verifier reused. No native conte

## Takeaways
D必须保留全部251机会，未知不补零；I只能使用原固定可配对支持，不能删除97个未匹配母事件。
单仓需按各臂真实保存退出重算占用，不能只比较保留下来的赢家。
正的收益变化也可能只是少亏；财务账本一致性不是策略有正期望或实盘成交保证。
不自动部署，也不把原生15分钟比较说成纯采样时钟实验。

### Execution gap
Plain Python top-down execution is not Jupyter-kernel execution. Minimum nbformat4.5 structure and code compilation are checked; full nbformat schema validation is not run. nbformat, nbclient and ipykernel are unavailable; no dependencies were installed.

完整Jupyter验证应在已有依赖的隔离环境运行
`python -m jupyter nbconvert --execute --to notebook --inplace path/to/native_exit_audit.ipynb`。
本轮不安装依赖；三格普通Python不是Jupyter内核或完整schema验证。